# Melbourne Water + Soil Data
## Initial Pre-processing (Shival)

#### Purpose

This notebook focuses on the initial data cleaning and preparation of the Melbourne Water Pipeline and Soil datasets for Project 3B.

The main goals are to:

- load and inspect the two raw datasets:
  - `Water_Supply_Main_Pipelines.csv`
  - `Soil-Readings-Melbourne.csv`
- understand the structure, data types, and data quality of each dataset
- clean and standardise key fields such as dates, numeric values, text fields, and IDs
- identify useful columns for later analysis and modelling our intial datasets
- check how the two datasets can be merged
- create a clean and consistent base dataset for the next stages of the project

In [4]:
import pandas as pd

In [5]:
#Initial check of current datasets to see what may need to be removed

#Load datasets into variables for cleaning and examination
water = pd.read_csv("../data/Water_Supply_Main_Pipelines.csv")
soil = pd.read_csv("../data/Soil-Readings-Melbourne.csv")

#print datasets to list and then proceed to print the first 5 rows of every column within said datasets.
print(water.columns.tolist())
print(soil.columns.tolist())

print(water.head())
print(soil.head())

['MAIN_LINE_TYPE', 'MAIN_CLASS', 'MAIN_NAME', 'Shape', 'MATERIAL', 'PIPE_LENGTH', 'PIPE_WIDTH', 'SOIL_TYPE', 'DATE_RELINED', 'DATE_OF_CONSTRUCTION', 'SOURCE', 'METHOD_OF_CAPTURE', 'DATE_CAPTURED', 'DATE_LAST_UPDATED', 'SERVICE_STATUS', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39']
['Local_Time', 'Site_Name', 'date', 'time', 'Probe_Measure', 'Soil_Value', 'Unit']
  MAIN_LINE_TYPE MAIN_CLASS                       MAIN_NAME Shape MATERIAL  \
0         MAINBG          T  ST GEORGES RD REPLACEMENT MAIN   NaN     MSCL   
1         MAINAG          S       CARDINIA CREEK MINI HYDRO   NaN     MSCL   
2         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M   NaN     MS

## Cleaning
### Water Pipeline Dataset

In [6]:
#Remove unecessary unnamed columns

#Removes empty unnamed columns within water dataset
water = water.loc[:, ~water.columns.str.contains("^Unnamed")]

#Keeps relevant columns
water = water[
    [
        "MAIN_LINE_TYPE",
        "MAIN_CLASS",
        "MAIN_NAME",
        "Shape",
        "MATERIAL",
        "PIPE_LENGTH",
        "PIPE_WIDTH",
        "SOIL_TYPE",
        "DATE_RELINED",
        "DATE_OF_CONSTRUCTION",
        "SOURCE",
        "METHOD_OF_CAPTURE",
        "DATE_CAPTURED",
        "DATE_LAST_UPDATED",
        "SERVICE_STATUS"
    ]
    ].copy()

#convert 'DATE_OF_CONSTRUCTION' to datetime for easier readability.
water["DATE_OF_CONSTRUCTION"] = pd.to_datetime(
    water["DATE_OF_CONSTRUCTION"], 
    format="%m/%d/%Y %I:%M:%S %p", #since our dataset contains a non standard format (MM/DD/YYYY + timestamp), we need to specify exact format for datetime to avoid runtime warnings
    errors="coerce" #this command serves to ensure that pandas doesn't crash when coming across invalid dates
)

#We can assume that the actual age of all of the varrying pipes within this data set are ages relative to their established date. Thus,
#we can simply subtract their build date from current year (2026) to estimate how old they are. We'll fit this data into a 'PIPE_AGE' column
water["PIPE_AGE"] = 2026 - water["DATE_OF_CONSTRUCTION"].dt.year

print(water.head())
print(water.isnull().sum())


  MAIN_LINE_TYPE MAIN_CLASS                       MAIN_NAME Shape MATERIAL  \
0         MAINBG          T  ST GEORGES RD REPLACEMENT MAIN   NaN     MSCL   
1         MAINAG          S       CARDINIA CREEK MINI HYDRO   NaN     MSCL   
2         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M   NaN     MSCL   
3         MAINBG          S                 TYABB RESERVOIR   NaN       MS   
4         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M   NaN     MSCL   

   PIPE_LENGTH  PIPE_WIDTH SOIL_TYPE DATE_RELINED DATE_OF_CONSTRUCTION  \
0         2.72        1400       NaN          NaN           2017-12-01   
1         1.88         225       NaN          NaN           2017-12-18   
2         1.74         225      SAND          NaN           2011-08-26   
3       131.30         100       NaN          NaN           2017-02-22   
4         5.26         225      SAND          NaN           2011-08-26   

           SOURCE METHOD_OF_CAPTURE           DATE_CAPTURED  \
0  AS CONSTRUCTED      

### Soil Dataset

In [7]:
#The cleaning stage for the soil dataset is mostly the same as the water dataset, we'll begin by removing unammed columns
soil = soil.loc[:, ~soil.columns.str.contains("^Unnamed")]

print(soil.columns.tolist())
print(soil.isnull().sum())

['Local_Time', 'Site_Name', 'date', 'time', 'Probe_Measure', 'Soil_Value', 'Unit']
Local_Time       928
Site_Name          0
date             928
time             928
Probe_Measure      0
Soil_Value         0
Unit               0
dtype: int64


In [8]:
#using the above metric, we'll then verify the columns we'd like to keep and rename them to a more standardised naming convention
soil = soil.rename(columns={
    "Local_Time": "LOCAL_TIME",
    "Site_Name": "SITE_NAME",
    "date": "DATE",
    "time": "TIME",
    "Probe_Measure": "PROBE_MEASURE",
    "Soil_Value": "SOIL_VALUE",
    "Unit": "UNIT"
})

print(soil.head())
print(soil.isnull().sum())

                  LOCAL_TIME            SITE_NAME      DATE      TIME  \
0  2023-03-03T08:00:00+11:00  Princes Park Lawn 5  3/3/2023   8:00:00   
1  2023-03-03T08:00:00+11:00  Princes Park Lawn 5  3/3/2023   8:00:00   
2  2023-03-03T10:00:00+11:00  Princes Park Lawn 5  3/3/2023  10:00:00   
3  2023-03-03T06:00:00+11:00  Princes Park Lawn 5  3/3/2023   6:00:00   
4  2023-03-03T10:00:00+11:00  Princes Park Lawn 5  3/3/2023  10:00:00   

           PROBE_MEASURE  SOIL_VALUE  UNIT  
0  Soil Moisture 50cm #0       36.21  %VWC  
1  Soil Moisture 60cm #0       46.91  %VWC  
2  Soil Moisture 60cm #0       46.91  %VWC  
3  Soil Moisture 70cm #0       56.90  %VWC  
4  Soil Moisture 70cm #0       56.90  %VWC  
LOCAL_TIME       928
SITE_NAME          0
DATE             928
TIME             928
PROBE_MEASURE      0
SOIL_VALUE         0
UNIT               0
dtype: int64


## Dataset preparation for using the Soil Dataset as a feature to compliment the water dataset

In [12]:
#Since a direct merge isn't the best option given the difference of the datasets, we're instead going to go for an approach where
#we filter the soil data to use it as a supporting dataset for our Melbourne Water Main Dataset
soil_moisture = soil[
    soil["PROBE_MEASURE"].str.contains("Moisture", case=False, na=False)
    ]

#it's also important we convert the soil values itself to a numeric value for easier analysis and use in models
soil_moisture["SOIL_VALUE"] = pd.to_numeric(
    soil_moisture["SOIL_VALUE"], errors="coerce"
)

#for the best result, we'll use the mean function to calcuate the average soil moisture of the soil moisture column in order to get a single value 
#we can use for testing
avg_moisture = soil_moisture["SOIL_VALUE"].mean()
print("Average Soil Moisture:", avg_moisture)

Average Soil Moisture: 30.591046075676182


We can now utilise the above value to compliment our water dataset during our Model

In [13]:
#We'll now add the value to our water dataset for usage later as a mean soil feature to each pipe in the dataset
water["AVG_SOIL_MOISTURE"] = avg_moisture

In [14]:
print(water.head())

  MAIN_LINE_TYPE MAIN_CLASS                       MAIN_NAME Shape MATERIAL  \
0         MAINBG          T  ST GEORGES RD REPLACEMENT MAIN   NaN     MSCL   
1         MAINAG          S       CARDINIA CREEK MINI HYDRO   NaN     MSCL   
2         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M   NaN     MSCL   
3         MAINBG          S                 TYABB RESERVOIR   NaN       MS   
4         BYPASS          S  CARDINIA-DAND'G-NOTTING HILL M   NaN     MSCL   

   PIPE_LENGTH  PIPE_WIDTH SOIL_TYPE DATE_RELINED DATE_OF_CONSTRUCTION  \
0         2.72        1400       NaN          NaN           2017-12-01   
1         1.88         225       NaN          NaN           2017-12-18   
2         1.74         225      SAND          NaN           2011-08-26   
3       131.30         100       NaN          NaN           2017-02-22   
4         5.26         225      SAND          NaN           2011-08-26   

           SOURCE METHOD_OF_CAPTURE           DATE_CAPTURED  \
0  AS CONSTRUCTED      